# One-feature baseline and image measurements

Needs the image files

1. The nested one-feature baseline on the same frames and folds
   as P1, written to `outputs/runs/`.
2. Per-recording intensity values behind Figure 2.
3. The frame differences(Patras only).

A dataset whose folder is missing is skipped.

In [ ]:
import os
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from src import data
from src.baseline import frame_features, nested_baseline
from src.run import load_config
from src.stats import per_fold

PATRAS_ROOT = "../data/patras"
PUCV_ROOT = "../data/pucv"
TABLES = "../outputs/tables"
RUNS = "../outputs/runs"

configs = {"patras": load_config("../configs/patras.yaml", data_root=PATRAS_ROOT),
           "pucv": load_config("../configs/pucv.yaml", data_root=PUCV_ROOT)}
available = {ds: os.path.isdir(cfg["data_root"]) for ds, cfg in configs.items()}
print(available)

## Nested baseline

In [ ]:
def run_baseline(ds):
    cfg = configs[ds]
    manifest = pd.read_csv(f"../outputs/manifests/{ds}/P1.csv")
    frames = data.frames_from_ids(cfg, manifest.sample_id)
    feats = pd.DataFrame([frame_features(f[0], cfg["body_threshold"]) for f in frames])
    df = pd.concat([manifest.reset_index(drop=True), feats], axis=1)
    df["label"] = [f[1] for f in frames]

    out = nested_baseline(df, cfg["threshold"])
    out.to_csv(f"{RUNS}/{ds}/baseline_preds.csv", index=False)
    per_fold(out).to_csv(f"{RUNS}/{ds}/baseline_folds.csv", index=False)
    print(ds, "feature chosen per fold:", out.groupby("fold").feature.first().value_counts().to_dict())
    return out


for ds in configs:
    if available[ds]:
        run_baseline(ds)

## Per-recording values

One value per recording: the statistic of each frame, averaged over all 50
frames. p05 for Patras, the median for PUCV-DTF.

In [ ]:
def per_recording(ds, stat):
    cfg = configs[ds]
    recs = data.list_recordings(cfg)
    subjects = sorted({s for s, _ in recs} - set(cfg["excluded"]))
    rows = []
    for s in subjects:
        vals = [np.mean([frame_features(f, cfg["body_threshold"])[stat] for f in data.read_recording(recs[(s, a)])])
                for a in (cfg["sober"], cfg["intoxicated"])]
        rows.append(dict(subject=s, sober=vals[0], intoxicated=vals[1]))
    table = pd.DataFrame(rows)
    print(f"{ds}: {stat} rises after drinking in {(table.intoxicated > table.sober).sum()} of {len(table)}")
    return table


if available["patras"]:
    per_recording("patras", "p05").to_csv(f"{TABLES}/patras_p05_per_recording.csv", index=False)
if available["pucv"]:
    per_recording("pucv", "p50").to_csv(f"{TABLES}/pucv_p50_per_recording.csv", index=False)

## Frame differences (Patras)

Mean absolute difference over the pixels present in both frames, for three
kinds of pair: frames of one burst, the same person sober and intoxicated,
and two different people.

In [ ]:
def compare(a, b):
    m = (a > 0) & (b > 0)
    return np.abs(a - b)[m].mean()


if available["patras"]:
    cfg = configs["patras"]
    recs = data.list_recordings(cfg)
    subjects = sorted({s for s, _ in recs} - set(cfg["excluded"]))
    rng = np.random.default_rng(0)
    pairs = {"same burst": [], "same person": [], "two people": []}

    for s in subjects:
        sober = data.read_recording(recs[(s, cfg["sober"])])
        drunk = data.read_recording(recs[(s, cfg["intoxicated"])])
        pairs["same burst"] += [compare(sober[0], sober[j]) for j in (1, 10, 25, 49)]
        pairs["same person"] += [compare(sober[i], drunk[i]) for i in range(0, 50, 10)]
    for s in subjects:
        other = rng.choice([o for o in subjects if o != s])
        pairs["two people"].append(compare(data.read_recording(recs[(s, cfg["sober"])])[0],
                                           data.read_recording(recs[(other, cfg["sober"])])[0]))

    diffs = pd.DataFrame([dict(pair=k, mean_abs_diff=np.mean(v), n=len(v)) for k, v in pairs.items()])
    diffs.to_csv(f"{TABLES}/patras_frame_differences.csv", index=False)
    print(diffs.round(2).to_string(index=False))